# SpectraShift Week 5: aggregate and approve Week 6
Use CPU with Internet off. Attach the newest source, Week 5 pilots, and all three Week 5 seed datasets.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week5.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 5 source bundle, found {bundles}'
    source_work = Path('/tmp/spectrashift-week5-source')
    if source_work.exists():
        shutil.rmtree(source_work)
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 5 source tree found'
projects.sort(key=lambda path: (0 if 'spectrashift-source' in str(path) else 1, len(str(path))))
PROJECT = projects[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

WORK = Path('/kaggle/working/spectrashift-week5-complete')
WORK.mkdir(parents=True, exist_ok=True)

seed_candidates = sorted(INPUT.rglob('week5_seed*_summary.json'))
pilot_candidates = sorted(INPUT.rglob('week5_pilot_summary.json'))
print({'seed_summary_candidates': [str(path) for path in seed_candidates],
       'pilot_summary_candidates': [str(path) for path in pilot_candidates]})

seed_groups = {}
for path in seed_candidates:
    payload = json.loads(path.read_text())
    if payload.get('week5_seed_complete') and payload.get('evaluation_labels_loaded') is False:
        seed_groups.setdefault(int(payload['seed']), []).append(path)
assert set(seed_groups) == {17, 29, 43}, f'Expected complete seeds 17, 29, 43; found {sorted(seed_groups)}'
seed_summaries = []
for seed in (17, 29, 43):
    paths = seed_groups[seed]
    hashes = {hashlib.sha256(path.read_bytes()).hexdigest() for path in paths}
    assert len(hashes) == 1, f'Conflicting summaries found for seed {seed}: {paths}'
    seed_summaries.append(paths[0])

valid_pilots = []
for path in pilot_candidates:
    payload = json.loads(path.read_text())
    if (payload.get('week5_pilots_complete') and payload.get('week5_final_approved')
            and payload.get('evaluation_labels_loaded') is False):
        valid_pilots.append(path)
assert valid_pilots, 'No approved Week 5 pilot summary was found'
pilot_hashes = {hashlib.sha256(path.read_bytes()).hexdigest() for path in valid_pilots}
assert len(pilot_hashes) == 1, f'Conflicting approved pilot summaries found: {valid_pilots}'
PILOT_SUMMARY = valid_pilots[0]
print({'selected_seed_summaries': [str(path) for path in seed_summaries],
       'selected_pilot_summary': str(PILOT_SUMMARY)})


In [ ]:
from spectrashift.train.week5 import aggregate_week5

summary = aggregate_week5(seed_summaries, PILOT_SUMMARY, WORK / 'week5_run_summary.json')
print(json.dumps(summary, indent=2))
assert summary['week5_complete'] and summary['week6_approved']
assert summary['run_count'] == 45 and summary['evaluation_labels_loaded'] is False
